# RSNA Knee Abnormality Detection - データ探索・解説ノートブック

本ノートブックでは、**RSNA Knee Abnormality Detection** コンペティションの問題設定、および提供されている各データファイル（`train_series.csv`, `train.csv`, `test_series.csv`, `test.csv`, `sample_submission.csv`）の構造と統計について分かりやすく解説および出力検証を行います。

## 1. 問題設定の概要

- **タスク**: 膝のMRI検査（1つの `StudyInstanceUID`）から、臨床的に重要な**12種類の膝病変・異常**を自動検出・予測するマルチラベル分類タスク。
- **入力データ**: 3D MRI画像ボリューム（複数方向スライス）
- **出力**: 12種類の病変それぞれが存在する確率（0.0 ~ 1.0）
- **評価指標**: 12ターゲットの **Macro-Averaged ROC-AUC**
- **12種類の病変ターゲット**:
  1. `ACL` (前十字靭帯損傷)
  2. `MCL` (内側側副靭帯損傷)
  3. `Medial Meniscus` (内側半月板異常)
  4. `Lateral Meniscus` (外側半月板異常)
  5. `Medial OA` (内側変形性膝関節症)
  6. `Lateral OA` (外側変形性膝関節症)
  7. `PF OA` (膝蓋大腿関節症)
  8. `Effusion` (関節液貯留)
  9. `Synovitis` (滑膜炎)
  10. `Baker's` (ベーカー嚢腫)
  11. `Contusion` (骨挫傷)
  12. `Fracture` (骨折)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ターゲット一覧
TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

print('ライブラリの読み込み完了。')

## 2. `train_series.csv` の解読と出力

`train_series.csv` は、各検査（`StudyInstanceUID`）に含まれる**画像シリーズ（撮影断面や強調モード）のメタデータ対応表**です。

- **`StudyInstanceUID`**: 検査ID（患者1人分のMRI検査一式）
- **`SeriesInstanceUID`**: シリーズID（特定方向の断面・撮像シーケンス）
- **`Anatomical_Plane`**: 断面の方向（`Sagittal`: 側面 / `Coronal`: 正面 / `Axial`: 水平）
- **`Fluid_Sensitive` / `Fat_Suppression`**: 水分強調・脂肪抑制フラグ（1: あり, 0: なし）

In [ ]:
# train_series.csv の読み込み
df_series = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv')
print(f"総行数 (シリーズ数): {len(df_series):,}")
print(f"ユニーク検査数 (StudyInstanceUID): {df_series['StudyInstanceUID'].nunique():,}")

print("\n--- 先頭5行 ---")
display(df_series.head())

print("\n--- 1検査あたりのシリーズ数分布 ---")
series_per_study = df_series.groupby('StudyInstanceUID')['SeriesInstanceUID'].count()
print(series_per_study.describe())

print("\n--- 断面(Plane)別シリーズ数 ---")
print(df_series['Anatomical_Plane'].value_counts())

print("\n--- 断面 × 水分強調 の組み合わせ ---")
print(pd.crosstab(df_series['Anatomical_Plane'], df_series['Fluid_Sensitive']))

## 3. `train.csv` の解読と出力

`train.csv` には、検査ID、**放射線科医の読影レポートテキスト (`Report`)**、および手動付与された正解ラベルが含まれています。

- **データ数の特徴**:
  - 全 4,407 件の検査すべてに `Report` テキストが存在します。
  - 手動で正解ラベルが付与されているのは**わずか 58 件**です（残り 4,349 件は `NaN`）。
  - そのため、学習前処理として **NLP/LLM等で `Report` から擬似正解ラベルを作成・付与する** アプローチが取られます。

In [ ]:
# train.csv の読み込み
df_train = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv')
print(f"総検査数: {len(df_train):,}")
print(f"Reportテキストが存在する数: {df_train['Report'].notnull().sum():,}")

# 全ラベルが埋まっている行数
labeled_count = df_train[TARGET_COLS].notnull().all(axis=1).sum()
print(f"ラベルが完全に付与されている検査数: {labeled_count} / {len(df_train)}")

print("\n--- Report（読影所見テキスト）のサンプル ---")
sample_report = df_train['Report'].dropna().iloc[0]
print(sample_report[:300] + "...")

print("\n--- 58件の正解ラベルにおける陽性（1.0）件数と陽性率 ---")
df_labeled = df_train.dropna(subset=TARGET_COLS)
label_stats = pd.DataFrame({
    'Positive (1)': (df_labeled[TARGET_COLS] == 1.0).sum(),
    'Negative (0)': (df_labeled[TARGET_COLS] == 0.0).sum(),
    'Positive Rate (%)': ((df_labeled[TARGET_COLS] == 1.0).mean() * 100).round(2)
})
display(label_stats)

## 4. 複数病変の同時発生（併発・マルチラベル）の検証

1人の患者に対して、**複数の病変が同時に発生（併発）しているか**を検証します。

In [ ]:
# 各検査ごとの陽性病変数
df_labeled['pos_count'] = df_labeled[TARGET_COLS].sum(axis=1)

print("--- 1人の患者における同時発生病変数の分布 ---")
pos_dist = df_labeled['pos_count'].value_counts().sort_index()
display(pos_dist.to_frame('患者数'))

# 複数併発の具体例
print("\n--- 複数病変が同時発生している具体的な組み合わせ例 ---")
for idx, row in df_labeled[df_labeled['pos_count'] > 1].head(5).iterrows():
    pos_diseases = [t for t in TARGET_COLS if row[t] == 1.0]
    print(f"StudyID: {row['StudyInstanceUID']}")
    print(f"  同時発生病変 ({len(pos_diseases)}個): {pos_diseases}")

## 5. `test.csv` と `test_series.csv` の解読と出力

テストフェーズ（評価・提出時）で使用するデータ構造の解説です。

- **`test.csv`**: 評価対象の `StudyInstanceUID` 一覧（※テスト時には `Report` テキストや病名ラベルは存在せず、`StudyInstanceUID` のみが与えられます）。
- **`test_series.csv`**: テスト用データの各検査に含まれる画像シリーズのメタデータ。
- **`sample_submission.csv`**: 提出フォーマット。12ターゲットそれぞれの予測確率を記述して出力します。

In [ ]:
# test.csv の読み込み
df_test = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/test.csv')
print("--- test.csv 内容 ---")
display(df_test)

# test_series.csv の読み込み
df_test_series = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv')
print(f"\n--- test_series.csv (全 {len(df_test_series)} シリーズ) ---")
display(df_test_series)

# sample_submission.csv の読み込み
df_sub = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv')
print("\n--- sample_submission.csv (提出用フォーマットサンプル) ---")
display(df_sub)

## 6. まとめとパイプライン構築への示唆

1. **データ構造の理解**:
   - `train_series.csv` / `test_series.csv`: 1検査あたり平均5.5個の画像シリーズ（Sagittal, Coronal, Axial断面）。
   - `train.csv`: 4,407件の読影テキスト (`Report`) と58件の正解ラベル。
   - `test.csv`: 評価対象の `StudyInstanceUID` 一覧（画像のみから予測する）。
2. **タスクの性質**:
   - 86%以上の患者で複数病変が併発する**マルチラベル問題**。
3. **前処理・モデル戦略**:
   - **ステップ1**: `Report` テキストからNLP/ルールベースで全データの疑似ラベル（Pseudo Label）を抽出・補完する。
   - **ステップ2**: `train_series.csv` のメタデータを元に主要断面（Sagittal/Coronal/Axial）のスライス画像を抽出し、画像モデル（DINOv2や2.5D CNN/ViT）に入力して12個の確率を出力・学習させる。
   - **ステップ3**: 学習済み画像モデルを使って、`test.csv` / `test_series.csv` の画像から12ターゲットの予測確率を計算し、`submission.csv` として出力する。